# Barranquilla MOPAC: benchmarks para el artículo

Mide la velocidad y la exactitud de todas las rutas GPU contra la CPU (el mismo binario con `MOPAC_NOGPU=1`) y
hace las figuras y tablas del artículo:

| Bloque | Qué mide | Tiempo (A100) |
|---|---|---|
| MOZYME | 1SCF de proteínas y ADN (642–6689 átomos) y cúmulos de agua (hasta 21 156 átomos); gradientes; 7 métodos; COSMO | ~1 h |
| Uso real | optimización de crambina (100 ciclos) y de 1AKE; dinámica NVE/NVT; FORCE / FORCETS contra referencias | ~1 h 15 min |
| Sólidos | SCF convencional: superceldas de Si (64–1728 átomos) y 10 celdas más; perfil por etapas; FORCE y DRC del diamante; moléculas con SCF convencional (RHF, UHF, COSMO, C.I.) | ~1 h 15 min |
| QM/MM | OpenMM + MOPAC: regiones QM de 16, 138 y 642 átomos (y 1231 opcional) en agua; tripsina–BPTI con y sin QM-PME | ~40 min |

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU A100* (o H100), con RAM alta si está
disponible. La CPU de referencia es la de la misma máquina de Colab.

**Si la sesión se corta:** los resultados se guardan en Google Drive (`USE_DRIVE = True`). Vuelva a abrir el
notebook, ejecute las celdas 1–4 y luego los bloques que faltan; las corridas ya hechas se leen y no se repiten.
Para rehacer una corrida, borre su carpeta en `runs/`.

**Resultados:** `data/*.csv` (una fila por corrida), `figures/*.pdf|png`, `tables/*.tex|csv`, `summary.md` y
`env.json` (hardware y versiones). La última celda descarga un zip.

## 1. Configuración

In [ ]:
BRANCH = 'paper-bench'        # rama o etiqueta del código (para el artículo: la etiqueta de la versión)
RESULTS_NAME = 'a100_run1'    # carpeta de resultados (otra GPU u otra corrida: otro nombre)
USE_DRIVE = True              # guardar en Google Drive (MyDrive/mopac_paper_bench/RESULTS_NAME) para retomar
REPEATS = 3                   # repeticiones de cada corrida en GPU (la figura usa la mediana)
QUICK = False                 # True: un caso pequeño por experimento (prueba del notebook, ~15 min)

# bloques (celdas 5-8)
RUN_MOZYME = True             # mozyme_sp, mozyme_grad, methods, cosmo
RUN_REAL_USE = True           # opt, md, thermo
RUN_SOLIDS = True             # solids, solids_profile, solids_force, solids_drc, conv_mol
RUN_QMMM = True               # qmmm, qmmm_enzyme

# límites de las corridas en CPU (la GPU corre siempre todos los tamaños)
CPU_MAX_ATOMS = 7000          # MOZYME en 1 núcleo: hasta 1AKE (6689 átomos, ~10 min) y el agua de 2000 moléculas
SOLIDS_CPU_MAX_ATOMS = 800    # SCF convencional en todos los núcleos: hasta las celdas de 768 átomos
SI_SIZES = '2,3,4,5,6'        # superceldas de Si n x n x n (8 n^3 átomos: 64, 216, 512, 1000, 1728)
CPU_LONG = False              # True: también la optimización de 1AKE en CPU (varias horas)
QMMM_LARGE = False            # True: también ubiquitina completa (1231 átomos) como región QM
ENZYME_STEPS = 2000           # pasos de QM/MM de tripsina-BPTI por variante (0.5 fs)

## 2. GPU, CPU y carpeta de resultados

In [ ]:
import os, re, subprocess, sys, time, shutil, json
from pathlib import Path

def run(cmd, check=True, env=None):
    # Run a shell command and stream its output to the cell
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
cpu = subprocess.run("lscpu | grep 'Model name' | sed 's/.*: *//'", shell=True, capture_output=True,
                     text=True).stdout.strip()
print(f'GPU: {gpu}\nCPU: {cpu}, {os.cpu_count()} núcleos lógicos')
CC = gpu.split(',')[-1].strip().replace('.', '')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/mopac_paper_bench') / RESULTS_NAME
else:
    OUT = Path('/content/mopac_paper_bench') / RESULTS_NAME
OUT.mkdir(parents=True, exist_ok=True)
print('resultados en', OUT)

## 3. Compilar MOPAC (programa y biblioteca) con OpenBLAS

OpenBLAS da a la referencia de CPU un BLAS optimizado y con hilos (con el BLAS de referencia de Ubuntu el SCF
convencional en CPU sería varias veces más lento y la comparación injusta). ~10 minutos.

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libopenblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
run(f'git clone -q --depth 1 -b {BRANCH} https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short')
t0 = time.time()
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || (tail -40 /content/cmake.log; exit 1)')
run(f'cmake --build {BUILD} --target mopac --parallel {os.cpu_count()} > /content/build.log 2>&1 '
    f'|| (grep -i -B2 -A6 error /content/build.log | head -80; exit 1)')
MOPAC = BUILD / 'mopac'
LIBMOPAC = next(BUILD.glob('libmopac.so*'))
run(f'ldd {MOPAC} | grep -i -E "blas|lapack"', check=False)
print(f'compilado en {time.time() - t0:.0f} s: {MOPAC}, {LIBMOPAC}')

## 4. OpenMM y CuPy (para QM/MM) y el lanzador de los experimentos

In [ ]:
if RUN_QMMM:
    # one CuPy, built for the CUDA of this runtime (Colab may already have cupy-cuda13x; a second CuPy package
    # for another CUDA breaks the kernels it compiles); if CuPy still fails its PME test, QM-PME runs its grid
    # with NumPy on the CPU (slower, same result)
    m = re.search(r'release (\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True,
                                                  text=True).stdout)
    cupy_pkg = f'cupy-cuda{m.group(1) if m else 12}x'
    other = [p for p in subprocess.run('pip list 2>/dev/null', shell=True, capture_output=True, text=True).stdout.split()
             if p.startswith('cupy-cuda') and p != cupy_pkg]
    if other:
        run('pip uninstall -y -q ' + ' '.join(other), check=False)
    run(f"pip install -q 'openmm[cuda12]==8.6.1' {cupy_pkg} 2>&1 | tail -2", check=False)
    run(f"{sys.executable} -c \"import openmm as m; print('OpenMM', m.__version__, "
        f"[m.Platform.getPlatform(i).getName() for i in range(m.Platform.getNumPlatforms())])\"", check=False)
    run(f"cd {SRC}/scripts/qmmm && {sys.executable} -c \"from qmmm_pme import gpu_available; "
        f"print('QM-PME grid on the GPU (CuPy):', gpu_available())\"", check=False)

BENCH = (f'{sys.executable} -u {SRC}/scripts/paper/paper_bench.py --mopac {MOPAC} --libmopac {LIBMOPAC} '
         f'--out {OUT} --repeats {REPEATS} --cpu-max-atoms {CPU_MAX_ATOMS} '
         f'--solids-cpu-max-atoms {SOLIDS_CPU_MAX_ATOMS} --si-sizes {SI_SIZES} --enzyme-steps {ENZYME_STEPS}'
         + (' --quick' if QUICK else '') + (' --cpu-long' if CPU_LONG else '') + (' --qmmm-large' if QMMM_LARGE else ''))

def bench(exps):
    t0 = time.time()
    run(f'{BENCH} --exp {exps}', check=False)
    print(f'--- {exps}: {(time.time() - t0) / 60:.1f} min')

bench('env')

## 5. MOZYME: proteínas, ADN y cúmulos de agua

- `mozyme_sp`: 1SCF, CPU (1 núcleo: MOZYME no tiene hilos) y GPU (`REPEATS` veces)
- `mozyme_grad`: 1SCF GRADIENTS, diferencia de calores y de gradientes
- `methods`: MNDO, AM1, PM3, RM1, PM6, PM6-D3H4 y PM7 en crambina y ADN
- `cosmo`: agua implícita (EPS=78.4)

In [ ]:
if RUN_MOZYME:
    bench('mozyme_sp,mozyme_grad,methods,cosmo')

## 6. Uso real: optimización, dinámica, frecuencias

- `opt`: crambina 100 ciclos (CPU y GPU) y 1AKE 50 ciclos (GPU); calores CPU y GPU en la geometría final
- `md`: crambina NVE a 300 K (CPU y GPU), NVT con termostato de Bussi y 1AKE NVE (GPU)
- `thermo`: FORCE de chignolina (4 procesos en la GPU) y FORCETS de crambina, contra referencias de CPU guardadas
  (la CPU tardó 84 min y 7.2 h; no se repiten aquí)

In [ ]:
if RUN_REAL_USE:
    bench('opt,md,thermo')

## 7. Sólidos y SCF convencional

- `solids`: 1SCF GRADIENTS de superceldas de Si y de 10 celdas (diamante, grafeno, MgO+agua, ZnO, cuarzo,
  rutilo, urea), CPU con todos los núcleos frente a GPU, calores y gradientes
- `solids_profile`: dónde se va el tiempo en las celdas de 512–768 átomos
- `solids_force`, `solids_drc`: frecuencias y dinámica del diamante
- `conv_mol`: crambina RHF, UHF y COSMO, hojuela de grafeno C192H38, gradiente C.I. de chignolina

In [ ]:
if RUN_SOLIDS:
    bench('solids,solids_profile,solids_force,solids_drc,conv_mol')

## 8. QM/MM con OpenMM

- `qmmm`: cadena lateral de Tyr (16 átomos QM, MOPAC en CPU como hace por defecto bajo 100 átomos), chignolina
  (138) y crambina (642) en agua TIP3P; MOPAC en GPU y en CPU; prueba NVE
- `qmmm_enzyme`: tripsina–BPTI (2PTC, ~46 000 átomos): solo cut-off frente a QM-PME, y el error de las fuerzas
  frente al cut-off

In [ ]:
if RUN_QMMM:
    bench('qmmm,qmmm_enzyme')

## 9. Figuras, tablas y resumen

In [ ]:
run(f'{sys.executable} {SRC}/scripts/paper/paper_figures.py {OUT}', check=False)
from IPython.display import Image, Markdown, display
display(Markdown((OUT / 'summary.md').read_text()))
for f in sorted((OUT / 'figures').glob('*.png')):
    print(f.name)
    display(Image(str(f), width=900))

## 10. Descargar

In [ ]:
# zip ligero: datos, figuras, tablas y resumen (las salidas completas de MOPAC quedan en Drive, runs/)
tmp = Path('/content/_paper_light') / RESULTS_NAME
shutil.rmtree(tmp.parent, ignore_errors=True)
tmp.mkdir(parents=True)
for sub in ('data', 'figures', 'tables'):
    if (OUT / sub).exists():
        shutil.copytree(OUT / sub, tmp / sub)
for f in ('summary.md', 'env.json'):
    if (OUT / f).exists():
        shutil.copy2(OUT / f, tmp / f)
# logs of the failed runs and of the QM/MM runs (small), to diagnose them
for f in list((OUT / 'runs').rglob('result_failed.json')) + list((OUT / 'runs').rglob('log.txt')) + \
        list((OUT / 'runs').rglob('reference_check.log')):
    dst = tmp / 'logs' / f.relative_to(OUT / 'runs')
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(f, dst)
    if f.name == 'result_failed.json':
        for g in f.parent.glob('*.stdout'):
            shutil.copy2(g, dst.parent / g.name)
light = shutil.make_archive(f'/content/paper_bench_{RESULTS_NAME}', 'zip', tmp.parent)
print(light)
from google.colab import files
files.download(light)